# QVerse — Introduction to Quantum Computing & Programming
        ## Week 10: Oracles and the Deutsch–Jozsa Algorithm

        **Level:** Beginner  
        **Recommended study time:** 2–4 hours  
        **Prerequisites:** Weeks 1–9

        ### Learning objectives
        - Recognize state preparation–oracle–interference–measurement structure.
- Understand an oracle as reversible problem encoding.
- Implement simple constant and balanced Deutsch–Jozsa oracles.
- Interpret the final measurement pattern.

        ---
        **How to use this notebook**

        1. Read the short theory sections.
        2. Make a prediction before running each guided experiment.
        3. Run and modify the code.
        4. Complete every **TODO** exercise.
        5. Finish the reflection section in your own words.

        The goal is not to memorize syntax. The goal is to connect **quantum idea → circuit → result → explanation**.

In [ ]:
# Run this only if your environment does not have the required packages.
# In a terminal, the preferred setup is:
# python -m pip install "qiskit[visualization]>=2.5" matplotlib numpy

# In a fresh Colab notebook you can instead uncomment:
# %pip install "qiskit[visualization]>=2.5" matplotlib numpy -q

## 1. Oracle thinking

Many introductory algorithms can be decomposed into:
1. prepare a useful input state;
2. apply an **oracle** that encodes the problem;
3. create interference;
4. measure and classically interpret the result.

Deutsch–Jozsa studies a promised Boolean function $f:\{0,1\}^n\to\{0,1\}$ that is either:
- **constant:** same output for every input;
- **balanced:** outputs 0 for half the inputs and 1 for half.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler

def dj_oracle(n, kind="constant0", mask=None):
    oracle = QuantumCircuit(n + 1, name="Oracle")
    target = n

    if kind == "constant0":
        pass
    elif kind == "constant1":
        oracle.x(target)
    elif kind == "balanced":
        if mask is None:
            mask = [1] * n
        if len(mask) != n or not any(mask):
            raise ValueError("balanced mask must contain at least one 1")
        for i, use_bit in enumerate(mask):
            if use_bit:
                oracle.cx(i, target)
    else:
        raise ValueError("Unknown oracle kind")

    return oracle

## 2. Deutsch–Jozsa driver

In [ ]:
def deutsch_jozsa(n, oracle):
    qc = QuantumCircuit(n + 1, n)

    # Ancilla starts in |1>
    qc.x(n)

    # Create |+> on input qubits and |-> on ancilla
    qc.h(range(n + 1))

    qc.append(oracle.to_gate(), range(n + 1))

    # Interfere input register
    qc.h(range(n))

    # Measure only input register
    qc.measure(range(n), range(n))
    return qc

for oracle in [
    dj_oracle(3, "constant0"),
    dj_oracle(3, "constant1"),
    dj_oracle(3, "balanced", [1, 0, 1]),
]:
    qc = deutsch_jozsa(3, oracle)
    counts = StatevectorSampler(seed=8).run([qc], shots=200).result()[0].data.c.get_counts()
    print(oracle.name, counts)

## 3. Decision rule

In the ideal algorithm:
- measuring `000...0` means **constant**;
- obtaining a nonzero bitstring means **balanced**.

The important beginner-level lesson is not the proof. It is how phase information created by the oracle is converted into a measurable pattern by interference.

## 4. Inspect a balanced example

In [ ]:
oracle = dj_oracle(3, "balanced", [1, 0, 1])
qc = deutsch_jozsa(3, oracle)
qc.draw("mpl")

## Core exercises
1. Classify several hand-written truth tables as constant, balanced, or neither.
2. Run the driver with `constant0` and `constant1` oracles.
3. Construct at least three different nonzero balanced masks and verify the classification.
4. Trace the algorithm conceptually: explain what state preparation, oracle, interference, and measurement each accomplish.

In [ ]:
# TODO: Write your solutions here.
# Add extra code cells when useful.

## Optional stretch challenge
For `n=4`, generate every nonzero linear mask used by this teaching oracle. Verify that the same Deutsch–Jozsa driver classifies all of them as balanced.

In [ ]:
# OPTIONAL TODO: Attempt the stretch challenge here.

## Weekly reflection
- What is an oracle?
- Why is reversibility important in circuit-based quantum computing?
- What role does interference play after the oracle?

## Submission checklist
- [ ] I made at least one prediction before executing a circuit.
- [ ] All guided examples run.
- [ ] I completed the core exercises.
- [ ] I explained the important output rather than only displaying it.
- [ ] My notebook is readable from top to bottom.